# Notes on Volatility Surface

### Notes

### Questions

- SANOS, then variational autoencoder (simulating stock and option surface)

### Potential Questions

- regime detection from change in volatility surface
- train generative model on the 500 equities, then test on S&P500

### Important Concepts

**Neural SDE**
- A stochastic differential equation $\small dY_t = \mu(\theta)Y_t \ dt + \sigma(\theta) Y_t \ dW_t$ where $\mu(\theta)$ and $\sigma(\theta)$ are neural network, fit to data by gradient descent. 
- Neural SDE require a lot of data for good training, and both parameters $\mu(\theta)$ and $\sigma(\theta)$ get fitted well differently. 
- For example, to get a good estimate of the diffusion $\sigma(\theta)$, it might be sufficient to increase the number of data by getting more frequent data (intraday), since the quadratic variation will converge. On the other hand, the drift $\mu(\theta)$ gets a better estimation by increasing the timelaps (get older data), which is not always easily doable. A good practice might be to set the initial drift to the linear regression slope, and ask the network to slightly adapt.


**Autoencoder** 
- An autoencoder (undercomplete) is a feedforward neural-network that tries to have the same inputs as outputs. The specialty of undercomplete autoencoder is to be used as a dimension reduction algorithm, that is, a 'middle' hidden layer has smaller dimension than the input/output. The idea is similar to PCA (if using linear activation function and MSE loss function, undercomplete autoencoder converges to PCA)
$$x(\mathbb{R}^5) \xrightarrow{encode} h(\mathbb{R}^{32}) \xrightarrow{encode} z(\mathbb{R}^2) \xrightarrow{decode}\hat h(\mathbb{R}^{32}) \xrightarrow{decode} \hat x(\mathbb{R}^5)$$


**Variational Autoencoder (VAE)**
- An autoencoder that outputs a distribution. Sample from the distribution before decoding (usually Gaussian)

- for exmaple, VAE: 5 features (input/output dimension), 32 neurons in each hidden layer, 2 latent variables (bottleneck size)
$$x(\mathbb{R}^5) \to h(\mathbb{R}^{32}) \to \big(\mu, \log\sigma^2\big)(\mathbb{R}^2,\mathbb{R}^2) \xrightarrow{\text{sample}} z(\mathbb{R}^2) \to \hat h(\mathbb{R}^{32}) \to \hat x(\mathbb{R}^5)$$

- sample is : $z = \mu + \sigma \epsilon$, $\epsilon \text{~} N(0,1)$

- Basically, the VAE compressed the repsentation of $x$ into $z$ (of lower dimension) and pushes it (by the KL loss function) the collection of $z$ to look like $\mathcal{N}(0,1)$

- Then, to simulate, simply sample from $\mathcal{N}(0,1)$ and decode it

- The KL loss function: For two probability distributions $P, Q$ over the same space:
- $\mathrm{KL}(P | Q) = \mathbb{E}_{z\sim P}\left[\ln\frac{P(z)}{Q(z)}\right]$, measures how different $P$ is from $Q$: how much probability mass $P$ places where $Q$ says it shouldn't be. $\mathrm{KL}(P|Q) \ge 0$ always, with equality iff $P = Q$. Not symmetric ($\mathrm{KL}(P|Q) \ne \mathrm{KL}(Q|P)$ in general) — not a true distance/metric.

- For the VAE, specifically:

$$P = q(z\mid x) = \mathcal{N}(\mu(x), \sigma^2(x)) \qquad\text{(encoder's output for a given }x\text{)}$$
$$Q = p(z) = \mathcal{N}(0, I) \qquad\text{(fixed shared prior, same for every }x\text{)}$$

### SANOS Paper

**Overview**
- The SANOS paper creates a smooth arbitrage free volatility surface.


**Research Orientation**
- introduce a dyamic volatility surface simulator. Use the paper "Deep Hedging: Learning to Remove the Drift," Risk, Feb 2022, to remove the drift and be able to efficiently simulate the volatility surface, with no calendar spread
1. Implement SANOS's static LP/DLV fitting on real SPX data
2. Fit SANOS day-by-day over a historical window and extract time series of $\Sigma_t$ and empirically characterize it (statistics, autcorrelation, cross-sectional dependance in strike)
3. Train a model (AR(1), GARCH), then train a generative model (autoencoder to Neural SDE, or VAE), and verify arbitrage (calendar and dynamic)
4. Potentially remove drift

Notes: Autoencoder to Neural SDE: *Since $\Sigma_j^i \geq 0$ is the only constraint SANOS's DLV parametrization needs, design the autoencoder's decoder output layer with a positivity-enforcing activation (softplus, exp, or square), then every decoded output — regardless of what the latent code $z_t$ or the Neural SDE does upstream — is automatically a valid, smooth, arbitrage-free SANOS surface. Static arbitrage-freedom for free, by construction*

- Multi-asset modeling